## EE8223 Deep Learning Project

##End-to-End Preprocessing and Fine-Tuning Pipeline for Wav2Vec2-Based ASVspoof Detection (to be used for IEMOCAP embeddings generation)

#Student: Jason Yip


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Preprocessing and Splitting ASVspoof Dataset for Training and Validation

### Overview
This script preprocesses the ASVspoof dataset, extracts labeled audio files, and splits them into balanced training and validation datasets. It ensures a minimum proportion of bonafide samples in the validation set while maintaining an overall balanced distribution for training. Key steps include file extraction, label parsing, sample allocation, and data serialization for future use.

### Detailed Steps

1. **Extract Audio Files from ZIP**:  
   - Unzips the audio files from a specified archive into a local directory if not already extracted.

2. **Parse Labels**:  
   - Reads the label file and associates file IDs with their respective labels (`bonafide` or `spoof`).

3. **File Matching and Filtering**:  
   - Ensures that only extracted files matching the labels are included in subsequent steps.

4. **Separate Bonafide and Spoof Samples**:  
   - Splits the matched files into bonafide and spoof categories for balanced dataset creation.

5. **Create Training and Validation Sets**:  
   - Allocates at least 25% bonafide samples in the validation set (total size: 1,000 samples) and balances the remaining files for the training set (total size: 9,000 samples).
   - Ensures the training and validation sets are shuffled to maintain randomness.

6. **Log File Creation**:  
   - Generates a detailed log file listing file paths and labels for both training and validation sets, along with summary statistics for transparency and reproducibility.

7. **Save Processed Data**:  
   - Serializes the training and validation sets into pickle files for reuse without repeating the preprocessing steps.

## Features

- **Balanced and Reproducible Splits**: Ensures balanced training and validation datasets with reproducible results.
- **Fair Validation Proportion**: Guarantees a minimum proportion of bonafide samples in the validation set for fair evaluation.
- **Detailed Logging**: Provides a log file with file paths, labels, and dataset statistics for easy verification.
- **Efficient Processing**: Handles large-scale audio datasets efficiently with structured outputs.

This script is ideal for preparing ASVspoof data for machine learning applications, particularly in spoof voice detection, while maintaining an organized and traceable workflow.


In [ ]:
import os
import random
from zipfile import ZipFile
import pickle  # For saving train and validation data

# Paths
zip_path = "/content/drive/MyDrive/ASVspoof_10000_train_subset_NEW.zip"  # Path to ZIP file in Google Drive
extract_path = "/content/ASVspoof_10000_train_subset_NEW"  # Extracted audio files path
label_file_path = "/content/drive/MyDrive/asvspoof_files/ASVspoof2019.LA.cm.train.trn.txt"
log_file_path = "/content/drive/MyDrive/ASVspoof_preprocessing_log.txt"  # Log file path

# Step 1: Extract ZIP file
if not os.path.exists(extract_path):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Extracted ZIP file to: {extract_path}")
else:
    print(f"Files already extracted to: {extract_path}")

# Step 2: Parse labels from the label file
def parse_asvspoof_labels(label_file):
    labels = []
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            file_id = parts[1]  # File ID
            label = parts[-1]  # Label: bonafide or spoof type
            labels.append((file_id, label))
    return labels

# Load and parse labels
data = parse_asvspoof_labels(label_file_path)

# Check if extracted files match the labels in `label_file`
extracted_files = {os.path.splitext(filename)[0] for filename in os.listdir(extract_path) if filename.endswith('.flac')}
matched_files = [(file_id, label) for file_id, label in data if file_id in extracted_files]

# Separate bonafide and spoof samples
bonafide_samples = [item for item in matched_files if item[1] == "bonafide"]
spoof_samples = [item for item in matched_files if item[1] != "bonafide"]

print(f"Number of bonafide samples: {len(bonafide_samples)}")
print(f"Number of spoof samples: {len(spoof_samples)}")

# Ensure there are enough samples
if len(spoof_samples) == 0 or len(bonafide_samples) == 0:
    raise ValueError("No spoof or bonafide samples found in the specified files. Verify that the files exist.")

# Step 3: Allocate at least 25% bonafide in validation
val_bonafide_count = min(len(bonafide_samples), 250)  # At least 25% of validation is bonafide (250 samples)
val_spoof_count = 1000 - val_bonafide_count  # Remaining spoof for validation (750 samples)

val_bonafide = bonafide_samples[:val_bonafide_count]
val_spoof = spoof_samples[:val_spoof_count]

# Remaining files go to training
train_bonafide = bonafide_samples[val_bonafide_count:]
train_spoof = spoof_samples[val_spoof_count:]

# Adjust training set to ensure exactly 9,000 samples
required_train_bonafide_count = min(len(train_bonafide), 9000 - len(train_spoof))  # Balance to ensure total 9,000
required_train_spoof_count = 9000 - required_train_bonafide_count

train_bonafide = train_bonafide[:required_train_bonafide_count]
train_spoof = train_spoof[:required_train_spoof_count]

# Combine to form training and validation sets
train_data = train_bonafide + train_spoof
val_data = val_bonafide + val_spoof

# Shuffle the combined train and validation sets
random.shuffle(train_data)
random.shuffle(val_data)

# Detailed distribution of bonafide and spoof in training and validation
train_bonafide_count = sum(1 for _, label in train_data if label == "bonafide")
train_spoof_count = sum(1 for _, label in train_data if label != "bonafide")
val_bonafide_count = sum(1 for _, label in val_data if label == "bonafide")
val_spoof_count = sum(1 for _, label in val_data if label != "bonafide")

print(f"\nTraining Data Distribution:")
print(f"  Bonafide samples: {train_bonafide_count}")
print(f"  Spoof samples: {train_spoof_count}")
print(f"  Total: {len(train_data)}")

print(f"\nValidation Data Distribution:")
print(f"  Bonafide samples: {val_bonafide_count}")
print(f"  Spoof samples: {val_spoof_count}")
print(f"  Total: {len(val_data)}")

# Step 4: Log file paths and labels for verification
with open(log_file_path, "w") as log_file:
    log_file.write("ASVspoof Preprocessing Log\n")
    log_file.write("=" * 50 + "\n\n")

    # Log training set
    log_file.write("Training Set:\n")
    for file_id, label in train_data:
        file_path = os.path.join(extract_path, f"{file_id}.flac")
        log_file.write(f"File: {file_path}, Label: {label}\n")

    # Log validation set
    log_file.write("\nValidation Set:\n")
    for file_id, label in val_data:
        file_path = os.path.join(extract_path, f"{file_id}.flac")
        log_file.write(f"File: {file_path}, Label: {label}\n")

    # Summary statistics
    log_file.write("\nSummary:\n")
    log_file.write(f"Total training samples: {len(train_data)}\n")
    log_file.write(f" - Bonafide samples: {train_bonafide_count}\n")
    log_file.write(f" - Spoof samples: {train_spoof_count}\n")
    log_file.write(f"Total validation samples: {len(val_data)}\n")
    log_file.write(f" - Bonafide samples: {val_bonafide_count}\n")
    log_file.write(f" - Spoof samples: {val_spoof_count}\n")

print(f"\nLog saved to: {log_file_path}")

# Step 5: Save train_data and val_data for reusability
with open("/content/train_data.pkl", "wb") as f:
    pickle.dump(train_data, f)
with open("/content/val_data.pkl", "wb") as f:
    pickle.dump(val_data, f)
print("\nSaved train_data and val_data.")


Extracted ZIP file to: /content/ASVspoof_10000_train_subset_NEW
Number of bonafide samples: 2580
Number of spoof samples: 7420

Training Data Distribution:
  Bonafide samples: 2330
  Spoof samples: 6670
  Total: 9000

Validation Data Distribution:
  Bonafide samples: 250
  Spoof samples: 750
  Total: 1000

Log saved to: /content/drive/MyDrive/ASVspoof_preprocessing_log.txt

Saved train_data and val_data.


## Fine-Tuning Wav2Vec2 for ASVspoof Detection Using BCEWithLogitsLoss

### Overview
This script fine-tunes the pretrained Wav2Vec2 model for detecting spoofed speech in the ASVspoof dataset. It utilizes a custom `BCEWithLogitsLoss` function with class weighting to handle class imbalance effectively, particularly between bonafide and spoof classes. The implementation includes data preprocessing, model training, evaluation, and saving results in a structured format.

### Detailed Workflow

1. **Setup and Initialization**:  
   - Installs required libraries, checks GPU availability, and configures file paths for input, output, and logging.

2. **Load and Preprocess Data**:  
   - Loads preprocessed train and validation datasets from pickle files.  
   - Uses `librosa` for audio loading and Wav2Vec2Processor for feature extraction.

3. **Custom Dataset Class**:  
   - Implements an `IterableDataset` class to handle large-scale audio datasets and streamline the preprocessing pipeline.

4. **Define BCEWithLogitsLoss**:  
   - Utilizes a weighted binary cross-entropy loss function to address class imbalance by assigning a higher weight to the bonafide class.

5. **Metrics Computation**:  
   - Calculates evaluation metrics, including accuracy, precision, recall, and F1 scores for both bonafide and spoof classes.

6. **Custom Callback**:  
   - Implements a `TrainerCallback` to save the best epoch and its metrics during evaluation.

7. **Training Setup**:  
   - Configures the `TrainingArguments` for the Hugging Face `Trainer`, including learning rate, batch size, logging, and checkpointing.  
   - Enables early stopping and saving the best model based on the F1 score of the bonafide class.

8. **Fine-Tuning the Model**:  
   - Fine-tunes the Wav2Vec2 model using the Hugging Face `Trainer` API, with the `BCEWithLogitsTrainer` handling custom loss functions.

9. **Model and Processor Saving**:  
   - Saves the fine-tuned model, tokenizer, and processor to specified output paths for reuse.

10. **Logging and Confirmation**:  
    - Saves validation predictions, training logs, and the best epoch information for reproducibility and analysis.

### Key Features

- **Custom BCE Loss**: Tailored loss function to address class imbalance, ensuring fair training for minority classes.  
- **Class-Weighted Training**: Assigns a higher weight to bonafide samples to improve detection performance.  
- **Comprehensive Logging**: Outputs logs, metrics, and predictions for monitoring and debugging.  
- **Integration with Hugging Face Trainer**: Combines the flexibility of custom loss functions with the efficiency of the Trainer API.  
- **GPU Acceleration**: Fully utilizes available GPU resources for faster training.  
- **Structured Output**: Saves the fine-tuned model, processor, and best epoch information for seamless downstream tasks.

This script is ideal for researchers working on spoof speech detection using deep learning models, providing an efficient, modular, and reproducible framework for fine-tuning the Wav2Vec2 model on the ASVspoof dataset.


In [ ]:
!pip install datasets evaluate librosa

from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, TrainerCallback
import torch
import torch.nn as nn
import os
import json
import pickle
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np
import pandas as pd
import librosa

# Completely disable wandb logging
os.environ["WANDB_DISABLED"] = "true"

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
audio_folder = "/content/ASVspoof_10000_train_subset_NEW"
output_model_path = "/content/drive/MyDrive/Wav2Vec2_FineTuned_ASVspoof_bce"
checkpoints_path = "/content/drive/MyDrive/Wav2Vec2_Checkpoints_bce"
tensorboard_log_dir = os.path.join(checkpoints_path, "tensorboard_logs")
log_file_path = "/content/drive/MyDrive/ASVspoof_finetuning_log_bce.txt"
best_epoch_path = "/content/drive/MyDrive/best_epoch_bce.json"
processor_save_path = "/content/drive/MyDrive/Wav2Vec2_Processor_ASVspoof_bce"

# Ensure the checkpoint and TensorBoard log directories exist
os.makedirs(checkpoints_path, exist_ok=True)
os.makedirs(tensorboard_log_dir, exist_ok=True)

# Load the preprocessed train and validation data
with open("/content/train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
with open("/content/val_data.pkl", "rb") as f:
    val_data = pickle.load(f)

# Initialize processor and model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
model = Wav2Vec2ForSequenceClassification.from_pretrained("facebook/wav2vec2-large-960h", num_labels=1)
model = model.to(device)

# Save the processor once before training starts
processor.save_pretrained(processor_save_path)
print(f"Processor saved to {processor_save_path}")

# Create data collator with padding
data_collator = DataCollatorWithPadding(tokenizer=processor)

# Custom IterableDataset
class ASVDataset(torch.utils.data.IterableDataset):
    def __init__(self, data, audio_folder):
        self.data = data
        self.audio_folder = audio_folder

    def parse_audio(self, file_id, label):
        try:
            file_path = os.path.join(self.audio_folder, f"{file_id}.flac")
            audio_input, _ = librosa.load(file_path, sr=16000)
            input_values = processor(audio_input, sampling_rate=16000, return_tensors="pt").input_values
            label_tensor = torch.tensor(label).float().to(device)

            return {"input_values": input_values.squeeze(0), "labels": label_tensor, "file_id": file_id}
        except Exception as e:
            with open(log_file_path, "a") as log_file:
                log_file.write(f"Error processing file {file_id}: {e}\n")
            return None

    def __iter__(self):
        for file_id, label in self.data:
            sample = self.parse_audio(file_id, 1 if label == "bonafide" else 0)
            if sample is not None:
                yield sample

# Create IterableDataset objects
train_dataset = ASVDataset(train_data, audio_folder)
val_dataset = ASVDataset(val_data, audio_folder)

# Define class-weighted BCEWithLogitsLoss
class BCEWithLogitsTrainer(Trainer):
    def __init__(self, pos_weight, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(self.model.device))

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(input_values=inputs["input_values"])
        logits = outputs.logits.squeeze()
        loss = self.loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Define metrics function
def compute_metrics(pred, file_ids=None, epoch=None, threshold=0.5):
    logits, labels = pred
    probabilities = torch.sigmoid(torch.tensor(logits))
    predictions = (probabilities > threshold).cpu().numpy()
    labels = np.array(labels)

    if file_ids is not None and epoch is not None:
        df = pd.DataFrame({"file_id": file_ids, "predictions": predictions, "labels": labels})
        df.to_csv(f"{checkpoints_path}/validation_predictions_epoch_{epoch}.csv", index=False)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average=None, labels=[0, 1])

    return {
        "accuracy": accuracy,
        "precision_spoof": precision[0],
        "recall_spoof": recall[0],
        "f1_spoof": f1[0],
        "precision_bonafide": precision[1],
        "recall_bonafide": recall[1],
        "f1_bonafide": f1[1]
    }

# Custom callback to save best epoch and confirm
class SaveBestEpochCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            with open(best_epoch_path, "w") as f:
                json.dump({"epoch": state.epoch, "metrics": metrics}, f)
            print(f"[Confirmation] Best epoch information saved to: {best_epoch_path}")

# Training arguments
training_args = TrainingArguments(
    output_dir=checkpoints_path,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    max_steps=5630,
    logging_dir=tensorboard_log_dir,  # Specify the TensorBoard log directory here
    logging_steps=10,
    learning_rate=1e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_bonafide",
    greater_is_better=True,
)

# Define class weights, higher weight for bonafide class
pos_weight = 1.95

# Trainer with BCEWithLogitsLoss
trainer = BCEWithLogitsTrainer(
    pos_weight=pos_weight,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[SaveBestEpochCallback()],
)

# Train the model
trainer.train()

# Save final model
model.save_pretrained(output_model_path)
print(f"Fine-tuned Wav2Vec2 model saved to {output_model_path}")
print(f"Processor saved earlier to: {processor_save_path}")

# Confirmation for log file
print(f"Log saved to: {log_file_path}")

# Final confirmation of best epoch info
if os.path.exists(best_epoch_path):
    print(f"Best epoch information successfully saved to: {best_epoch_path}")
else:
    print("Warning: Best epoch information was not saved.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 18.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 18.0.0 which i

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-large-960h and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
max_steps is given, it will override any value giv

Processor saved to /content/drive/MyDrive/Wav2Vec2_Processor_ASVspoof_bce


Epoch,Training Loss,Validation Loss,Accuracy,Precision Spoof,Recall Spoof,F1 Spoof,Precision Bonafide,Recall Bonafide,F1 Bonafide
0,0.825600,0.812805,0.751000,0.750751,1.000000,0.857633,1.000000,0.004000,0.007968
1,0.675900,0.686344,0.650000,1.000000,0.533333,0.695652,0.416667,1.000000,0.588235
2,0.189700,0.359000,0.880000,1.000000,0.840000,0.913043,0.675676,1.000000,0.806452
3,0.078800,0.353653,0.908000,1.000000,0.877333,0.934659,0.730994,1.000000,0.844595
4,0.087100,0.296779,0.936000,1.000000,0.914667,0.955432,0.796178,1.000000,0.886525
5,0.075400,0.391518,0.919000,1.000000,0.892000,0.942918,0.755287,1.000000,0.860585
6,0.012600,0.282674,0.943000,1.000000,0.924000,0.960499,0.814332,1.000000,0.897666
7,0.012400,0.185104,0.966000,1.000000,0.954667,0.976808,0.880282,1.000000,0.936330
8,0.135200,0.309351,0.940000,1.000000,0.920000,0.958333,0.806452,1.000000,0.892857
9,0.054400,0.194759,0.964000,1.000000,0.952000,0.975410,0.874126,1.000000,0.932836


[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json
[Confirmation] Best epoch information saved to: /content/drive/MyDrive/best_epoch_bce.json